In [0]:
# Celula 1 (Notebook Gold): Parametros do ambiente e verificacao da tabela Silver de leituras
catalog = "workspace"
bronze_schema = f"`{catalog}`.bronze"
silver_schema = f"`{catalog}`.silver"
gold_schema = f"`{catalog}`.gold"

silver_leituras_table = f"{silver_schema}.leituras"

df_silver_leituras = spark.table(silver_leituras_table)
total_silver = df_silver_leituras.count()

print(f"Tabela lida: {silver_leituras_table}")
print(f"Total de linhas na Silver: {total_silver:,} (Esperado: 2,152,336)")
print("\nSchema da Silver de Leituras:")
df_silver_leituras.printSchema()

Tabela lida: `workspace`.silver.leituras
Total de linhas na Silver: 2,152,336 (Esperado: 2,152,336)

Schema da Silver de Leituras:
root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- data_leitura: date (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- leitura_anterior: integer (nullable = true)
 |-- leitura_atual: integer (nullable = true)
 |-- consumo_kwh: integer (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- flag_sinal_corrigido: boolean (nullable = true)
 |-- flag_troca_medidor: boolean (nullable = true)
 |-- flag_sem_leitura_anterior: boolean (nullable = true)
 |-- flag_sinal_nao_confirmado: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
# Celula 2 (Notebook Gold): Criacao da tabelafato Gold de Leituras Enriquecidas
df_silver = spark.table(f"{silver_schema}.leituras")
df_uc = spark.table(f"{bronze_schema}.unidades_consumidoras")

# Selecionamos os atributos chave de localizacao e rede da unidade consumidora
df_uc_attrs = df_uc.select(
    "id_uc",
    "classe",
    "subgrupo_tarifario",
    "alimentador",
    "municipio",
    "uf"
)

# Cruzamento (Join) da Silver de Leituras com as UCs
df_gold_enriquecida = (
    df_silver.alias("l")
    .join(df_uc_attrs.alias("uc"), on="id_uc", how="inner")
    .select(
        "l.id_leitura",
        "l.id_uc",
        "l.id_medidor",
        "l.data_leitura",
        "l.mes_referencia",
        "l.leitura_anterior",
        "l.leitura_atual",
        "l.consumo_kwh",
        "l.tipo_leitura",
        "uc.classe",
        "uc.subgrupo_tarifario",
        "uc.alimentador",
        "uc.municipio",
        "uc.uf",
        "l.flag_sinal_corrigido",
        "l.flag_troca_medidor",
        "l.flag_sem_leitura_anterior",
        "l.flag_sinal_nao_confirmado",
        "l.source_file",
        "l.ingestion_datetime"
    )
)

gold_table_enriquecida = f"{gold_schema}.f_leituras_enriquecidas"

(
    df_gold_enriquecida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_enriquecida)
)

print(f"[SUCESSO] Tabela Gold criada: {gold_table_enriquecida}")
print(f"Total de linhas gravadas: {spark.table(gold_table_enriquecida).count():,}")

[SUCESSO] Tabela Gold criada: `workspace`.gold.f_leituras_enriquecidas
Total de linhas gravadas: 2,212,620


In [0]:
# Celula 3 (Notebook Gold): Auditoria de duplicatas no cadastro e integridade referencial
from pyspark.sql import functions as F

df_silver_leituras = spark.table(f"{silver_schema}.leituras")
df_bronze_uc = spark.table(f"{bronze_schema}.unidades_consumidoras")
df_bronze_med = spark.table(f"{bronze_schema}.medidores")

COLUNAS_UC_NEGOCIO = [
    "id_uc", "cpf_cnpj", "nome_titular", "logradouro", "numero", "bairro",
    "municipio", "uf", "cep", "classe", "subgrupo_tarifario", "alimentador",
    "data_ligacao", "status", "latitude", "longitude",
]

total_uc_bruto = df_bronze_uc.count()
total_uc_sem_dup_exata = df_bronze_uc.dropDuplicates(COLUNAS_UC_NEGOCIO).count()
total_id_uc_distintos = df_bronze_uc.select("id_uc").distinct().count()

# Verificar se ha UCs nas leituras Silver que nao existem no cadastro (orfaos)
ucs_leituras = df_silver_leituras.select("id_uc").distinct()
ucs_cadastro = df_bronze_uc.select("id_uc").distinct()
ucs_orfas = ucs_leituras.subtract(ucs_cadastro).count()

print("=== AUDITORIA DE UNIDADES CONSUMIDORAS ===")
print(f"Total bruto na Bronze:             {total_uc_bruto:,}")
print(f"Apos dropDuplicates(colunas):      {total_uc_sem_dup_exata:,}")
print(f"Total de id_uc distintos:          {total_id_uc_distintos:,}")
print(f"UCs em silver.leituras sem cadastro (orfas): {ucs_orfas:,}")

# Se ainda houver mais linhas sem duplicata exata do que id_uc distintos, inspecionar divergencia
if total_uc_sem_dup_exata > total_id_uc_distintos:
    print("\n[AVISO] Existem id_uc repetidos com valores divergentes entre colunas:")
    ucs_duplicadas = (
        df_bronze_uc.dropDuplicates(COLUNAS_UC_NEGOCIO)
        .groupBy("id_uc")
        .count()
        .filter("count > 1")
    )
    print(f"Quantidade de id_uc com divergencia cadastral: {ucs_duplicadas.count():,}")
    df_bronze_uc.join(ucs_duplicadas.limit(3), on="id_uc").orderBy("id_uc").show(truncate=False)

=== AUDITORIA DE UNIDADES CONSUMIDORAS ===
Total bruto na Bronze:             46,461
Apos dropDuplicates(colunas):      45,561
Total de id_uc distintos:          45,201
UCs em silver.leituras sem cadastro (orfas): 0

[AVISO] Existem id_uc repetidos com valores divergentes entre colunas:
Quantidade de id_uc com divergencia cadastral: 360
+--------+-----------+------------------------+-------------------+------+----------------+-----------------------+----------+--------+-----------+------------------+-----------+------------+------+--------+---------+-------------------------+--------------------------+-----+
|id_uc   |cpf_cnpj   |nome_titular            |logradouro         |numero|bairro          |municipio              |uf        |cep     |classe     |subgrupo_tarifario|alimentador|data_ligacao|status|latitude|longitude|source_file              |ingestion_datetime        |count|
+--------+-----------+------------------------+-------------------+------+----------------+----------------

In [0]:
# Celula 4 (Notebook Gold): Saneamento da dimensao UC e recriacao de f_leituras_enriquecidas sem fan-out
from pyspark.sql import functions as F

df_silver = spark.table(f"{silver_schema}.leituras")
df_bronze_uc = spark.table(f"{bronze_schema}.unidades_consumidoras")

# 1. Padronizar atributos da UC (caixa alta, trim e normalizacao de UF) e deduplicar por id_uc
df_uc_limpa = (
    df_bronze_uc
    .select(
        F.trim(F.col("id_uc")).alias("id_uc"),
        F.upper(F.trim(F.col("classe"))).alias("classe"),
        F.upper(F.trim(F.col("subgrupo_tarifario"))).alias("subgrupo_tarifario"),
        F.upper(F.trim(F.col("alimentador"))).alias("alimentador"),
        F.initcap(F.trim(F.col("municipio"))).alias("municipio"),
        F.when(F.upper(F.trim(F.col("uf"))).isin("PE", "PERNAMBUCO"), F.lit("PE"))
         .otherwise(F.upper(F.trim(F.col("uf")))).alias("uf"),
        F.upper(F.trim(F.col("status"))).alias("status_uc")
    )
    .dropDuplicates(["id_uc"])
)

total_uc_limpa = df_uc_limpa.count()
print(f"Total de UCs unicas apos padronizacao: {total_uc_limpa:,} (Esperado: 45,201)")
assert total_uc_limpa == 45_201, f"Contagem de UCs diverge do esperado: {total_uc_limpa}"

# Conferir os dominios padronizados de classe, uf e status_uc
print("\nDominios padronizados na dimensao UC:")
for col_name in ["classe", "subgrupo_tarifario", "uf", "status_uc"]:
    valores = [r[0] for r in df_uc_limpa.select(col_name).distinct().orderBy(col_name).collect()]
    print(f" - {col_name}: {valores}")

# 2. Recriar a tabela Gold enriquecida mantendo a granularidade exata da Silver (2.152.336 linhas)
flag_anomalia = (
    F.col("l.flag_troca_medidor") |
    F.col("l.flag_sem_leitura_anterior") |
    F.col("l.flag_sinal_nao_confirmado")
)

df_gold_enriquecida = (
    df_silver.alias("l")
    .join(df_uc_limpa.alias("uc"), on="id_uc", how="inner")
    .select(
        "l.id_leitura",
        "l.id_uc",
        "l.id_medidor",
        "l.data_leitura",
        "l.mes_referencia",
        "l.leitura_anterior",
        "l.leitura_atual",
        "l.consumo_kwh",
        # Regra da Silver: para somar consumo confiavel na Gold, considerar apenas consumo_kwh > 0 (sem flag negativa)
        F.when(~flag_anomalia & (F.col("l.consumo_kwh") >= 0), F.col("l.consumo_kwh"))
         .otherwise(F.lit(None)).alias("consumo_kwh_valido"),
        "l.tipo_leitura",
        "uc.classe",
        "uc.subgrupo_tarifario",
        "uc.alimentador",
        "uc.municipio",
        "uc.uf",
        "uc.status_uc",
        "l.flag_sinal_corrigido",
        "l.flag_troca_medidor",
        "l.flag_sem_leitura_anterior",
        "l.flag_sinal_nao_confirmado",
        flag_anomalia.alias("possui_anomalia_consumo"),
        "l.source_file",
        "l.ingestion_datetime"
    )
)

gold_table_enriquecida = f"{gold_schema}.f_leituras_enriquecidas"

(
    df_gold_enriquecida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_enriquecida)
)

total_gold_gravado = spark.table(gold_table_enriquecida).count()
print(f"\n[SUCESSO] Tabela {gold_table_enriquecida} atualizada: {total_gold_gravado:,} linhas")
assert total_gold_gravado == 2_152_336, f"Erro de fan-out! Esperado 2.152.336, obtido {total_gold_gravado}"

Total de UCs unicas apos padronizacao: 45,201 (Esperado: 45,201)

Dominios padronizados na dimensao UC:
 - classe: ['CLASSE', 'COMERCIAL', 'INDUSTRIAL', 'PODER PUBLICO', 'PODER PÚBLICO', 'RESIDENCIAL', 'RURAL']
 - subgrupo_tarifario: ['A4', 'B1', 'B2', 'B3', 'SUBGRUPO_TARIFARIO']
 - uf: ['P.E.', 'PE', 'UF']
 - status_uc: ['ATIVA', 'STATUS']

[SUCESSO] Tabela `workspace`.gold.f_leituras_enriquecidas atualizada: 2,152,336 linhas


In [0]:
# Celula 5 (Notebook Gold): Correcao de dominios (cabecalho infiltrado, acentos e UF) e inspecao de municipio/tipo_leitura
from pyspark.sql import functions as F

df_silver = spark.table(f"{silver_schema}.leituras")
df_bronze_uc = spark.table(f"{bronze_schema}.unidades_consumidoras")

# 1. Remover linha de cabecalho infiltrada e padronizar dominios
df_uc_limpa = (
    df_bronze_uc
    .filter(F.upper(F.trim(F.col("id_uc"))) != "ID_UC")
    .select(
        F.trim(F.col("id_uc")).alias("id_uc"),
        F.when(
            F.upper(F.trim(F.col("classe"))).isin("PODER PUBLICO", "PODER PÚBLICO"),
            F.lit("PODER PÚBLICO")
        ).otherwise(F.upper(F.trim(F.col("classe")))).alias("classe"),
        F.upper(F.trim(F.col("subgrupo_tarifario"))).alias("subgrupo_tarifario"),
        F.upper(F.trim(F.col("alimentador"))).alias("alimentador"),
        F.initcap(F.trim(F.col("municipio"))).alias("municipio"),
        F.when(
            F.upper(F.trim(F.col("uf"))).isin("PE", "P.E.", "PERNAMBUCO"),
            F.lit("PE")
        ).otherwise(F.upper(F.trim(F.col("uf")))).alias("uf"),
        F.upper(F.trim(F.col("status"))).alias("status_uc")
    )
    .dropDuplicates(["id_uc"])
)

total_uc_limpa = df_uc_limpa.count()
print(f"Total de UCs reais apos remover cabecalho infiltrado: {total_uc_limpa:,} (Esperado: 45,200)")
assert total_uc_limpa == 45_200, f"Contagem inesperada: {total_uc_limpa}"

print("\nDominios corrigidos na dimensao UC:")
for col_name in ["classe", "subgrupo_tarifario", "uf", "status_uc", "municipio"]:
    valores = [r[0] for r in df_uc_limpa.select(col_name).distinct().orderBy(col_name).collect()]
    print(f" - {col_name} ({len(valores)}): {valores}")

tipos_leitura = [r[0] for r in df_silver.select("tipo_leitura").distinct().orderBy("tipo_leitura").collect()]
print(f" - tipo_leitura na Silver ({len(tipos_leitura)}): {tipos_leitura}")

# 2. Atualizar a tabela f_leituras_enriquecidas com os dominios limpos
flag_anomalia = (
    F.col("l.flag_troca_medidor") |
    F.col("l.flag_sem_leitura_anterior") |
    F.col("l.flag_sinal_nao_confirmado")
)

df_gold_enriquecida = (
    df_silver.alias("l")
    .join(df_uc_limpa.alias("uc"), on="id_uc", how="inner")
    .select(
        "l.id_leitura",
        "l.id_uc",
        "l.id_medidor",
        "l.data_leitura",
        "l.mes_referencia",
        "l.leitura_anterior",
        "l.leitura_atual",
        "l.consumo_kwh",
        F.when(~flag_anomalia & (F.col("l.consumo_kwh") >= 0), F.col("l.consumo_kwh"))
         .otherwise(F.lit(None)).alias("consumo_kwh_valido"),
        F.upper(F.trim(F.col("l.tipo_leitura"))).alias("tipo_leitura"),
        "uc.classe",
        "uc.subgrupo_tarifario",
        "uc.alimentador",
        "uc.municipio",
        "uc.uf",
        "uc.status_uc",
        "l.flag_sinal_corrigido",
        "l.flag_troca_medidor",
        "l.flag_sem_leitura_anterior",
        "l.flag_sinal_nao_confirmado",
        flag_anomalia.alias("possui_anomalia_consumo"),
        "l.source_file",
        "l.ingestion_datetime"
    )
)

gold_table_enriquecida = f"{gold_schema}.f_leituras_enriquecidas"

(
    df_gold_enriquecida.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_enriquecida)
)

total_gold = spark.table(gold_table_enriquecida).count()
print(f"\n[SUCESSO] Tabela {gold_table_enriquecida} regravada com {total_gold:,} linhas.")
assert total_gold == 2_152_336, f"Volume incorreto: {total_gold}"

Total de UCs reais apos remover cabecalho infiltrado: 45,200 (Esperado: 45,200)

Dominios corrigidos na dimensao UC:
 - classe (5): ['COMERCIAL', 'INDUSTRIAL', 'PODER PÚBLICO', 'RESIDENCIAL', 'RURAL']
 - subgrupo_tarifario (4): ['A4', 'B1', 'B2', 'B3']
 - uf (1): ['PE']
 - status_uc (1): ['ATIVA']
 - municipio (14): ['Abreu E Lima', 'Araçoiaba', 'Cabo De Santo Agostinho', 'Camaragibe', 'Igarassu', 'Ilha De Itamaracá', 'Ipojuca', 'Itapissuma', 'Jaboatão Dos Guararapes', 'Moreno', 'Olinda', 'Paulista', 'Recife', 'São Lourenço Da Mata']
 - tipo_leitura na Silver (2): ['ESTIMADA', 'REAL']

[SUCESSO] Tabela `workspace`.gold.f_leituras_enriquecidas regravada com 2,152,336 linhas.


In [0]:
# Celula 6 (Notebook Gold): Criacao da tabela consolidada agg_consumo_mensal
from pyspark.sql import functions as F

df_fato = spark.table(f"{gold_schema}.f_leituras_enriquecidas")

df_agg_mensal = (
    df_fato
    .groupBy(
        "mes_referencia",
        "municipio",
        "alimentador",
        "classe",
        "subgrupo_tarifario"
    )
    .agg(
        F.count("*").alias("qtd_leituras_total"),
        F.countDistinct("id_uc").alias("qtd_ucs_medidas"),
        F.count("consumo_kwh_valido").alias("qtd_leituras_validas"),
        F.coalesce(F.sum("consumo_kwh_valido"), F.lit(0)).alias("consumo_total_kwh"),
        F.round(F.avg("consumo_kwh_valido"), 2).alias("consumo_medio_kwh"),
        F.min("consumo_kwh_valido").alias("consumo_min_kwh"),
        F.max("consumo_kwh_valido").alias("consumo_max_kwh"),
        F.sum(F.when(F.col("tipo_leitura") == "REAL", 1).otherwise(0)).alias("qtd_leituras_reais"),
        F.sum(F.when(F.col("tipo_leitura") == "ESTIMADA", 1).otherwise(0)).alias("qtd_leituras_estimadas"),
        F.round(
            F.sum(F.when(F.col("tipo_leitura") == "ESTIMADA", 1).otherwise(0)) * 100.0 / F.count("*"),
            2
        ).alias("pct_leituras_estimadas"),
        F.sum(F.col("flag_sinal_corrigido").cast("int")).alias("qtd_flag_sinal_corrigido"),
        F.sum(F.col("flag_troca_medidor").cast("int")).alias("qtd_flag_troca_medidor"),
        F.sum(F.col("flag_sem_leitura_anterior").cast("int")).alias("qtd_flag_sem_leitura_anterior"),
        F.sum(F.col("flag_sinal_nao_confirmado").cast("int")).alias("qtd_flag_sinal_nao_confirmado"),
        F.sum(F.col("possui_anomalia_consumo").cast("int")).alias("qtd_anomalias_negativas")
    )
    .withColumn("processed_datetime", F.current_timestamp())
    .orderBy("mes_referencia", "municipio", "alimentador", "classe", "subgrupo_tarifario")
)

gold_table_mensal = f"{gold_schema}.agg_consumo_mensal"

(
    df_agg_mensal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_mensal)
)

df_check = spark.table(gold_table_mensal)
print(f"[SUCESSO] Tabela criada: {gold_table_mensal}")
print(f"Total de linhas consolidadas (grupos): {df_check.count():,}")

# Reconciliacao rapida com a tabela fato
resumo = df_check.agg(
    F.sum("qtd_leituras_total").alias("soma_leituras"),
    F.sum("qtd_leituras_validas").alias("soma_validas"),
    F.sum("qtd_anomalias_negativas").alias("soma_anomalias"),
    F.sum("consumo_total_kwh").alias("soma_consumo_kwh")
).first().asDict()

print("Resumo global reconciliado:", resumo)
display(df_check.limit(10))

[SUCESSO] Tabela criada: `workspace`.gold.agg_consumo_mensal
Total de linhas consolidadas (grupos): 26,736
Resumo global reconciliado: {'soma_leituras': 2152336, 'soma_validas': 2145654, 'soma_anomalias': 6682, 'soma_consumo_kwh': 888655666}


mes_referencia,municipio,alimentador,classe,subgrupo_tarifario,qtd_leituras_total,qtd_ucs_medidas,qtd_leituras_validas,consumo_total_kwh,consumo_medio_kwh,consumo_min_kwh,consumo_max_kwh,qtd_leituras_reais,qtd_leituras_estimadas,pct_leituras_estimadas,qtd_flag_sinal_corrigido,qtd_flag_troca_medidor,qtd_flag_sem_leitura_anterior,qtd_flag_sinal_nao_confirmado,qtd_anomalias_negativas,processed_datetime
2022-07,Abreu E Lima,ALM-ABR-001,COMERCIAL,B3,60,60,60,38972,649.53,104,3136,46,14,23.33,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-001,INDUSTRIAL,A4,5,5,5,63466,12693.2,2753,30885,5,0,0.0,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-001,PODER PÚBLICO,B3,2,2,2,2265,1132.5,662,1603,2,0,0.0,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-001,RESIDENCIAL,B1,485,485,485,49986,103.06,26,402,402,83,17.11,1,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-001,RURAL,B2,7,7,7,1891,270.14,84,441,7,0,0.0,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-002,COMERCIAL,B3,19,19,19,10204,537.05,141,1684,13,6,31.58,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-002,INDUSTRIAL,A4,4,4,4,14822,3705.5,1785,6701,2,2,50.0,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-002,PODER PÚBLICO,B3,1,1,1,2086,2086.0,2086,2086,0,1,100.0,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-002,RESIDENCIAL,B1,161,161,161,16620,103.23,34,238,132,29,18.01,0,0,0,0,0,2026-10-31T13:21:31.826Z
2022-07,Abreu E Lima,ALM-ABR-002,RURAL,B2,3,3,3,928,309.33,173,577,2,1,33.33,0,0,0,0,0,2026-10-31T13:21:31.826Z


In [0]:
# Celula 7 (Notebook Gold): Criacao da tabela consolidada por Unidade Consumidora (agg_perfil_consumo_uc)
from pyspark.sql import functions as F

df_fato = spark.table(f"{gold_schema}.f_leituras_enriquecidas")

df_perfil_uc = (
    df_fato
    .groupBy(
        "id_uc",
        "municipio",
        "uf",
        "alimentador",
        "classe",
        "subgrupo_tarifario",
        "status_uc"
    )
    .agg(
        F.min("mes_referencia").alias("primeiro_mes_leitura"),
        F.max("mes_referencia").alias("ultimo_mes_leitura"),
        F.count("*").alias("qtd_meses_medidos"),
        F.countDistinct("id_medidor").alias("qtd_medidores_distintos"),
        F.count("consumo_kwh_valido").alias("qtd_leituras_validas"),
        F.coalesce(F.sum("consumo_kwh_valido"), F.lit(0)).alias("consumo_historico_total_kwh"),
        F.round(F.avg("consumo_kwh_valido"), 2).alias("consumo_medio_mensal_kwh"),
        F.min("consumo_kwh_valido").alias("consumo_min_mensal_kwh"),
        F.max("consumo_kwh_valido").alias("consumo_max_mensal_kwh"),
        F.sum(F.when(F.col("tipo_leitura") == "REAL", 1).otherwise(0)).alias("qtd_leituras_reais"),
        F.sum(F.when(F.col("tipo_leitura") == "ESTIMADA", 1).otherwise(0)).alias("qtd_leituras_estimadas"),
        F.round(
            F.sum(F.when(F.col("tipo_leitura") == "ESTIMADA", 1).otherwise(0)) * 100.0 / F.count("*"),
            2
        ).alias("pct_leituras_estimadas"),
        F.sum(F.col("flag_sinal_corrigido").cast("int")).alias("qtd_flag_sinal_corrigido"),
        F.sum(F.col("flag_troca_medidor").cast("int")).alias("qtd_trocas_medidor"),
        F.sum(F.col("flag_sem_leitura_anterior").cast("int")).alias("qtd_flag_sem_leitura_anterior"),
        F.sum(F.col("flag_sinal_nao_confirmado").cast("int")).alias("qtd_flag_sinal_nao_confirmado"),
        F.sum(F.col("possui_anomalia_consumo").cast("int")).alias("qtd_anomalias_negativas")
    )
    .withColumn("processed_datetime", F.current_timestamp())
)

gold_table_uc = f"{gold_schema}.agg_perfil_consumo_uc"

(
    df_perfil_uc.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_uc)
)

df_check_uc = spark.table(gold_table_uc)
total_ucs_gold = df_check_uc.count()

print(f"[SUCESSO] Tabela criada: {gold_table_uc}")
print(f"Total de UCs consolidadas: {total_ucs_gold:,} (Esperado: 45,200)")
assert total_ucs_gold == 45_000, f"Contagem de UCs inesperada: {total_ucs_gold}"

display(df_check_uc.orderBy(F.col("consumo_historico_total_kwh").desc()).limit(10))

[SUCESSO] Tabela criada: `workspace`.gold.agg_perfil_consumo_uc
Total de UCs consolidadas: 45,000 (Esperado: 45,200)


id_uc,municipio,uf,alimentador,classe,subgrupo_tarifario,status_uc,primeiro_mes_leitura,ultimo_mes_leitura,qtd_meses_medidos,qtd_medidores_distintos,qtd_leituras_validas,consumo_historico_total_kwh,consumo_medio_mensal_kwh,consumo_min_mensal_kwh,consumo_max_mensal_kwh,qtd_leituras_reais,qtd_leituras_estimadas,pct_leituras_estimadas,qtd_flag_sinal_corrigido,qtd_trocas_medidor,qtd_flag_sem_leitura_anterior,qtd_flag_sinal_nao_confirmado,qtd_anomalias_negativas,processed_datetime
UC138427,Recife,PE,ALM-REC-017,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2667042,55563.38,45299,64009,45,3,6.25,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC142495,Moreno,PE,ALM-MOR-002,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2462832,51309.0,43356,61294,47,1,2.08,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC105646,Olinda,PE,ALM-OLI-011,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,2,47,2384542,50734.94,42170,59520,27,21,43.75,0,1,0,0,1,2026-10-31T13:22:40.193Z
UC144926,Jaboatão Dos Guararapes,PE,ALM-JAB-001,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2362856,49226.17,42271,57284,47,1,2.08,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC112744,Recife,PE,ALM-REC-005,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2354891,49060.23,40923,59973,47,1,2.08,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC119025,Cabo De Santo Agostinho,PE,ALM-CAB-003,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2332397,48591.6,39840,57296,46,2,4.17,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC135155,Itapissuma,PE,ALM-ITA-001,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2149505,44781.35,37267,50639,30,18,37.5,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC130908,Recife,PE,ALM-REC-029,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2116795,44099.9,34723,53387,47,1,2.08,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC132442,Cabo De Santo Agostinho,PE,ALM-CAB-005,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2114864,44059.67,37176,53003,48,0,0.0,0,0,0,0,0,2026-10-31T13:22:40.193Z
UC118230,São Lourenço Da Mata,PE,ALM-SAO-004,INDUSTRIAL,A4,ATIVA,2022-07,2026-06,48,1,48,2091710,43577.29,34807,50967,46,2,4.17,0,0,0,0,0,2026-10-31T13:22:40.193Z


In [0]:
# Celula 8 (Notebook Gold): Testes automatizados de qualidade e reconciliacao cruzada (Silver <-> Gold)
from pyspark.sql import functions as F

df_silver = spark.table(f"{silver_schema}.leituras")
df_fato = spark.table(f"{gold_schema}.f_leituras_enriquecidas")
df_mensal = spark.table(f"{gold_schema}.agg_consumo_mensal")
df_uc = spark.table(f"{gold_schema}.agg_perfil_consumo_uc")

# 1. Reconciliacao de contagem de leituras
qtd_silver = df_silver.count()
qtd_fato = df_fato.count()
qtd_mensal = df_mensal.agg(F.sum("qtd_leituras_total")).first()[0]
qtd_uc = df_uc.agg(F.sum("qtd_meses_medidos")).first()[0]

# 2. Reconciliacao de consumo valido (kWh)
kwh_silver_valido = df_silver.filter(F.col("consumo_kwh") >= 0).agg(F.sum("consumo_kwh")).first()[0]
kwh_fato = df_fato.agg(F.sum("consumo_kwh_valido")).first()[0]
kwh_mensal = df_mensal.agg(F.sum("consumo_total_kwh")).first()[0]
kwh_uc = df_uc.agg(F.sum("consumo_historico_total_kwh")).first()[0]

# 3. Verificacao de ausencia de consumos negativos nas metricas validas
min_fato = df_fato.agg(F.min("consumo_kwh_valido")).first()[0]
min_mensal = df_mensal.agg(F.min("consumo_min_kwh")).first()[0]
min_uc = df_uc.agg(F.min("consumo_min_mensal_kwh")).first()[0]

# 4. Reconciliacao de anomalias negativas (6.215 + 114 + 353 = 6.682)
anom_fato = df_fato.agg(F.sum(F.col("possui_anomalia_consumo").cast("int"))).first()[0]
anom_mensal = df_mensal.agg(F.sum("qtd_anomalias_negativas")).first()[0]
anom_uc = df_uc.agg(F.sum("qtd_anomalias_negativas")).first()[0]

# 5. Verificacao de nulos nas dimensoes obrigatorias da Gold
dimensoes_obrigatorias = ["id_uc", "mes_referencia", "municipio", "uf", "alimentador", "classe", "subgrupo_tarifario"]
nulos_fato = sum(df_fato.filter(F.col(c).isNull()).count() for c in dimensoes_obrigatorias)

print("=== RELATORIO DE TESTES DE QUALIDADE DA CAMADA GOLD ===")
print(f"1. Contagem de leituras -> Silver: {qtd_silver:,} | Fato: {qtd_fato:,} | Mensal: {qtd_mensal:,} | UC: {qtd_uc:,}")
print(f"2. Consumo total (kWh)  -> Silver: {kwh_silver_valido:,} | Fato: {kwh_fato:,} | Mensal: {kwh_mensal:,} | UC: {kwh_uc:,}")
print(f"3. Menor consumo valido -> Fato: {min_fato} | Mensal: {min_mensal} | UC: {min_uc}")
print(f"4. Total de anomalias   -> Fato: {anom_fato:,} | Mensal: {anom_mensal:,} | UC: {anom_uc:,}")
print(f"5. Registos com dimensao nula na Gold: {nulos_fato}")

assert qtd_silver == qtd_fato == qtd_mensal == qtd_uc == 2_152_336, "Falha na reconciliacao de volume de leituras!"
assert kwh_silver_valido == kwh_fato == kwh_mensal == kwh_uc, "Falha na reconciliacao da soma de consumo (kWh)!"
assert min_fato >= 0 and min_mensal >= 0 and min_uc >= 0, "Consumo negativo detetado nas metricas validas!"
assert anom_fato == anom_mensal == anom_uc == 6_682, "Falha na reconciliacao das flags de anomalia!"
assert nulos_fato == 0, "Existem dimensoes nulas na camada Gold!"

print("\n[OK] Todos os testes automatizados de qualidade da camada Gold passaram sem erros.")

=== RELATORIO DE TESTES DE QUALIDADE DA CAMADA GOLD ===
1. Contagem de leituras -> Silver: 2,152,336 | Fato: 2,152,336 | Mensal: 2,152,336 | UC: 2,152,336
2. Consumo total (kWh)  -> Silver: 888,655,666 | Fato: 888,655,666 | Mensal: 888,655,666 | UC: 888,655,666
3. Menor consumo valido -> Fato: 8 | Mensal: 8 | UC: 8
4. Total de anomalias   -> Fato: 6,682 | Mensal: 6,682 | UC: 6,682
5. Registos com dimensao nula na Gold: 0

[OK] Todos os testes automatizados de qualidade da camada Gold passaram sem erros.


In [0]:
# Celula 9 (Notebook Gold): Publicacao do Dicionario de Dados auditavel da Camada Gold
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

schema_dicionario = StructType([
    StructField("tabela", StringType(), False),
    StructField("coluna", StringType(), False),
    StructField("tipo_dado", StringType(), False),
    StructField("definicao_metrica", StringType(), False),
    StructField("decisoes_limpeza_afetadas", StringType(), False),
])

dados_dicionario = [
    # 1. Tabela Fato Enriquecida: f_leituras_enriquecidas
    ("f_leituras_enriquecidas", "id_leitura", "string", "Identificador unico do registro de leitura.", "Deduplicacao de 21.523 copias exatas na Silver e remocao de 8.609 clones LTX com medidor inexistente."),
    ("f_leituras_enriquecidas", "id_uc", "string", "Identificador da Unidade Consumidora (1 leitura por par id_uc e mes_referencia).", "Integridade referencial validada com cadastro de UCs (deduplicado de 46.461 para 45.200 UCs reais apos remover 1 linha de cabecalho infiltrada, 900 duplicatas exatas e 360 divergencias de caixa alta/baixa)."),
    ("f_leituras_enriquecidas", "id_medidor", "string", "Identificador do medidor fisico vinculado a leitura.", "Filtro left_semi na Silver contra bronze.medidores descartando 8.609 leituras de medidores MD9 nao cadastrados."),
    ("f_leituras_enriquecidas", "data_leitura", "date", "Data efetiva da medicao convertida para padrao ISO (yyyy-MM-dd).", "Padronizacao via regex na Silver a partir de 4 formatos distintos (yyyy-MM-dd, dd/MM/yyyy, dd.MM.yyyy e MM-dd-yyyy)."),
    ("f_leituras_enriquecidas", "mes_referencia", "string", "Competencia mensal da leitura no formato AAAA-MM.", "Validado na Silver para coincidir exatamente com o ano e mes de data_leitura."),
    ("f_leituras_enriquecidas", "leitura_anterior", "integer", "Valor acumulado do medidor no ciclo anterior (em kWh).", "Marcadores de ausencia ('', 'N/A', 'NULL', '-1') convertidos para NULL antes do cast para inteiro na Silver."),
    ("f_leituras_enriquecidas", "leitura_atual", "integer", "Valor acumulado do medidor no ciclo atual (em kWh).", "Convertido de texto para inteiro na Silver sem ocorrencia de nulos."),
    ("f_leituras_enriquecidas", "consumo_kwh", "integer", "Consumo medido no mes (preserva valores negativos de anomalias para auditoria).", "Remocao de sufixo decimal '.0' e inversao de sinal apenas para as 1.334 leituras confirmadas por flag_sinal_corrigido; mantem diferenca de +/-1 kWh como arredondamento."),
    ("f_leituras_enriquecidas", "consumo_kwh_valido", "integer", "Consumo mensal saneado ( >= 0 ) utilizado para somas e medias oficiais na Gold.", "Anula (NULL) as 6.682 leituras negativas com flag de anomalia (6.215 troca de medidor, 114 sem leitura anterior e 353 sinal nao confirmado) para nao subestimar o consumo agregado."),
    ("f_leituras_enriquecidas", "tipo_leitura", "string", "Modalidade de coleta da leitura (REAL ou ESTIMADA).", "Padronizado em caixa alta e sem espacos."),
    ("f_leituras_enriquecidas", "classe", "string", "Classe de consumo da UC (RESIDENCIAL, COMERCIAL, INDUSTRIAL, RURAL, PODER PUBLICO).", "Unificacao de caixa alta e correcao de acentuacao ('PODER PUBLICO' para 'PODER PÚBLICO')."),
    ("f_leituras_enriquecidas", "subgrupo_tarifario", "string", "Subgrupo tarifario da UC (A4, B1, B2, B3).", "Padronizado em caixa alta apos remocao da linha de cabecalho infiltrada no CSV bruto."),
    ("f_leituras_enriquecidas", "alimentador", "string", "Codigo do alimentador eletrico ao qual a UC esta conectada.", "Padronizado em caixa alta a partir do cadastro de UCs."),
    ("f_leituras_enriquecidas", "municipio", "string", "Municipio de localizacao da Unidade Consumidora.", "Padronizado em formato InitCap (14 municipios da RMR/PE)."),
    ("f_leituras_enriquecidas", "uf", "string", "Unidade Federativa da UC (PE).", "Normalizado de 'PE', 'pe', 'P.E.' e 'Pernambuco' para 'PE'."),
    ("f_leituras_enriquecidas", "possui_anomalia_consumo", "boolean", "Indica se a leitura possui consumo negativo nao corrigido (True para 6.682 registros).", "Consolidacao logica de flag_troca_medidor OR flag_sem_leitura_anterior OR flag_sinal_nao_confirmado."),

    # 2. Tabela Agregada Mensal: agg_consumo_mensal
    ("agg_consumo_mensal", "qtd_leituras_total", "long", "Total de leituras registradas no mes por municipio, alimentador, classe e subgrupo.", "Inclui leituras validas e leituras com anomalia (soma global = 2.152.336)."),
    ("agg_consumo_mensal", "qtd_ucs_medidas", "long", "Quantidade de Unidades Consumidoras distintas medidas no grupo e mes.", "Como a Silver garante 1 leitura por (id_uc, mes_referencia), coincide com qtd_leituras_total."),
    ("agg_consumo_mensal", "qtd_leituras_validas", "long", "Quantidade de leituras com consumo valido (>= 0) consideradas nas metricas de kWh.", "Exclui as 6.682 leituras com anomalia negativa (soma global = 2.145.654)."),
    ("agg_consumo_mensal", "consumo_total_kwh", "long", "Soma do consumo de energia eletrica (kWh) do grupo no mes.", "Calculado exclusivamente sobre consumo_kwh_valido (soma global = 888.655.666 kWh) e sem fan-out de join."),
    ("agg_consumo_mensal", "consumo_medio_kwh", "double", "Media de consumo mensal (kWh) por leitura valida no grupo.", "Ignora registros com possui_anomalia_consumo = True para evitar deflacao artificial da media."),
    ("agg_consumo_mensal", "pct_leituras_estimadas", "double", "Percentual de leituras realizadas por estimativa (ESTIMADA / total * 100).", "Calculado sobre o total de leituras deduplicadas e sem clones LTX."),
    ("agg_consumo_mensal", "qtd_anomalias_negativas", "long", "Total de leituras negativas retidas para auditoria no grupo e mes.", "Soma de flag_troca_medidor, flag_sem_leitura_anterior e flag_sinal_nao_confirmado."),

    # 3. Tabela Agregada por UC: agg_perfil_consumo_uc
    ("agg_perfil_consumo_uc", "qtd_meses_medidos", "long", "Quantidade total de competencias mensais medidas no historico da UC.", "Reflete as 45.000 UCs que possuem leituras ativas entre 2022 e 2026 (200 UCs do cadastro nao possuem leituras)."),
    ("agg_perfil_consumo_uc", "qtd_medidores_distintos", "long", "Quantidade de medidores fisicos distintos que ja operaram na UC.", "Contabiliza apenas medidores validos existentes em bronze.medidores."),
    ("agg_perfil_consumo_uc", "consumo_historico_total_kwh", "long", "Consumo acumulado valido (kWh) da UC em todo o periodo de 2022 a 2026.", "Soma estrita de consumo_kwh_valido, reconciliada em 888.655.666 kWh."),
    ("agg_perfil_consumo_uc", "consumo_medio_mensal_kwh", "double", "Media mensal historica de consumo (kWh) da Unidade Consumidora.", "Calculada apenas sobre os meses sem anomalia negativa de leitura."),
    ("agg_perfil_consumo_uc", "qtd_trocas_medidor", "long", "Quantidade de meses em que a UC apresentou reset de leitura por troca de medidor.", "Contagem de flag_troca_medidor (consumo < 0 e leitura_anterior > leitura_atual)."),
]

df_dicionario = (
    spark.createDataFrame(dados_dicionario, schema=schema_dicionario)
    .withColumn("published_datetime", F.current_timestamp())
)

gold_table_dict = f"{gold_schema}.dicionario_dados_leituras"

(
    df_dicionario.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table_dict)
)

print(f"[SUCESSO] Dicionario de dados publicado em: {gold_table_dict}")
print(f"Total de definicoes documentadas: {spark.table(gold_table_dict).count()}")
display(spark.table(gold_table_dict))

[SUCESSO] Dicionario de dados publicado em: `workspace`.gold.dicionario_dados_leituras
Total de definicoes documentadas: 28


tabela,coluna,tipo_dado,definicao_metrica,decisoes_limpeza_afetadas,published_datetime
f_leituras_enriquecidas,id_leitura,string,Identificador unico do registro de leitura.,Deduplicacao de 21.523 copias exatas na Silver e remocao de 8.609 clones LTX com medidor inexistente.,2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,id_uc,string,Identificador da Unidade Consumidora (1 leitura por par id_uc e mes_referencia).,"Integridade referencial validada com cadastro de UCs (deduplicado de 46.461 para 45.200 UCs reais apos remover 1 linha de cabecalho infiltrada, 900 duplicatas exatas e 360 divergencias de caixa alta/baixa).",2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,id_medidor,string,Identificador do medidor fisico vinculado a leitura.,Filtro left_semi na Silver contra bronze.medidores descartando 8.609 leituras de medidores MD9 nao cadastrados.,2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,data_leitura,date,Data efetiva da medicao convertida para padrao ISO (yyyy-MM-dd).,"Padronizacao via regex na Silver a partir de 4 formatos distintos (yyyy-MM-dd, dd/MM/yyyy, dd.MM.yyyy e MM-dd-yyyy).",2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,mes_referencia,string,Competencia mensal da leitura no formato AAAA-MM.,Validado na Silver para coincidir exatamente com o ano e mes de data_leitura.,2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,leitura_anterior,integer,Valor acumulado do medidor no ciclo anterior (em kWh).,"Marcadores de ausencia ('', 'N/A', 'NULL', '-1') convertidos para NULL antes do cast para inteiro na Silver.",2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,leitura_atual,integer,Valor acumulado do medidor no ciclo atual (em kWh).,Convertido de texto para inteiro na Silver sem ocorrencia de nulos.,2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,consumo_kwh,integer,Consumo medido no mes (preserva valores negativos de anomalias para auditoria).,Remocao de sufixo decimal '.0' e inversao de sinal apenas para as 1.334 leituras confirmadas por flag_sinal_corrigido; mantem diferenca de +/-1 kWh como arredondamento.,2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,consumo_kwh_valido,integer,Consumo mensal saneado ( >= 0 ) utilizado para somas e medias oficiais na Gold.,"Anula (NULL) as 6.682 leituras negativas com flag de anomalia (6.215 troca de medidor, 114 sem leitura anterior e 353 sinal nao confirmado) para nao subestimar o consumo agregado.",2026-10-31T13:26:05.946Z
f_leituras_enriquecidas,tipo_leitura,string,Modalidade de coleta da leitura (REAL ou ESTIMADA).,Padronizado em caixa alta e sem espacos.,2026-10-31T13:26:05.946Z
